In [1]:
from google.colab import drive, auth
import os
import zipfile
import numpy as np
import pandas as pd
import rasterio
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
import pickle
from datetime import datetime

drive.mount('/content/drive')
auth.authenticate_user()

gpus = tf.config.list_physical_devices('GPU')
print(f"GPU: {gpus}")
assert len(gpus) > 0, "No GPU. Check runtime type."
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

Mounted at /content/drive
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# ── PATHS ──────────────────────────────────────────────
# CHANGED from phase2:
#   - Input: ZIP archives on Drive (not TFRecords on GCS)
#   - Model: proxy_cnn_final.keras (same model, no TPU)
#   - PCA  : final_pca.pkl (new — reduce 4096→128)
#   - Output keyed on PointID not ClusterID

TILES_DIR    = '/content/drive/MyDrive/Datamaraws_2025_Data/tiles_zipped'
MODEL_PATH   = '/content/drive/MyDrive/Models/proxy_cnn_final.keras'
PCA_PATH     = '/content/drive/MyDrive/Models/final_pca.pkl'
OUTPUT_DRIVE = '/content/drive/MyDrive/Datamaraws_2025_Data/dynamic_features_2025_prediction_points.csv'
CHECKPOINT   = '/content/drive/MyDrive/Datamaraws_2025_Data/feat_extract_2025_checkpoint.csv'

GCS_OUTPUT   = 'gs://tala-sentinel2-data/features/output_2025'

BATCH_SIZE   = 32

# Verify files exist
for label, path in [
    ('Tiles dir', TILES_DIR),
    ('Model',     MODEL_PATH),
    ('PCA',       PCA_PATH)
]:
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f"  {status}  {label}: {path}")

zips = sorted([f for f in os.listdir(TILES_DIR)
               if f.endswith('.zip')])
print(f"\nZIP archives found: {len(zips)}")
for z in zips:
    sz = os.path.getsize(os.path.join(TILES_DIR, z)) / 1e6
    print(f"  {z}  ({sz:.0f} MB)")

  ✓  Tiles dir: /content/drive/MyDrive/Datamaraws_2025_Data/tiles_zipped
  ✓  Model: /content/drive/MyDrive/Models/proxy_cnn_final.keras
  ✓  PCA: /content/drive/MyDrive/Models/final_pca.pkl

ZIP archives found: 11
  tiles_Aklan.zip  (2100 MB)
  tiles_Basilan.zip  (2142 MB)
  tiles_Benguet.zip  (5075 MB)
  tiles_Davao_Oriental.zip  (7898 MB)
  tiles_Ilocos_Norte.zip  (6710 MB)
  tiles_Kalinga.zip  (5781 MB)
  tiles_Maguindanao_del_Sur.zip  (4312 MB)
  tiles_NCR.zip  (7916 MB)
  tiles_Pampanga.zip  (4566 MB)
  tiles_Tawi_Tawi.zip  (1319 MB)
  tiles_Zamboanga_del_Norte.zip  (9086 MB)


In [5]:
# ── LOAD MODEL ─────────────────────────────────────────
# CHANGED from phase2:
#   - No TPU, no strategy.scope()
#   - Same feature_vector layer extraction
#   - Also load final_pca.pkl (not in phase2)

print("Loading proxy CNN feature extractor...")
full_model = load_model(MODEL_PATH)
feature_extractor = Model(
    inputs  = full_model.input,
    outputs = full_model.get_layer('feature_vector').output
)
print(f"Input  : {feature_extractor.input_shape}")
print(f"Output : {feature_extractor.output_shape}")

print("\nLoading PCA transformer...")
with open(PCA_PATH, 'rb') as f:
    pca = pickle.load(f)

N_COMPONENTS = pca.n_components_
print(f"PCA components : {N_COMPONENTS}")
print(f"PCA variance   : {pca.explained_variance_ratio_.sum():.4f}")
print("✓ Ready")

Loading proxy CNN feature extractor...
Input  : (None, 224, 224, 3)
Output : (None, 4096)

Loading PCA transformer...
PCA components : 256
PCA variance   : 0.9087
✓ Ready


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# ── BUILD TILE INDEX ───────────────────────────────────
# CHANGED from phase2:
#   - phase2 used tf.io.gfile.glob() on GCS TFRecord shards
#   - Here we scan ZIP files on Drive to build a flat index
#     of all (PointID, Quarter, zip_path, internal_path)
#   - This mirrors what phase2's feature_spec parsed from
#     TFRecord fields: cluster_id → PointID, quarter → Quarter

print("Scanning ZIP archives for tiles...")
tile_index = []   # list of (PointID, Quarter, zip_path, internal_tif_path)

for zip_fname in sorted(os.listdir(TILES_DIR)):
    if not zip_fname.endswith('.zip'):
        continue
    zip_path = os.path.join(TILES_DIR, zip_fname)
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            tif_files = sorted([
                n for n in z.namelist()
                if n.endswith('.tif')
            ])
            for tif_path in tif_files:
                # Parse: 00001/point_00001_2025_Q1.tif
                parts    = os.path.basename(tif_path)\
                             .replace('.tif', '').split('_')
                # parts: ['point', '00001', '2025', 'Q1']
                point_id = int(parts[1])
                quarter  = parts[3]     # Q1, Q2, Q3, Q4
                tile_index.append({
                    'PointID'  : point_id,
                    'Quarter'  : quarter,
                    'zip_path' : zip_path,
                    'tif_path' : tif_path
                })
    except zipfile.BadZipFile:
        print(f"  ERROR: corrupted ZIP — {zip_fname}")

df_index = pd.DataFrame(tile_index)
print(f"\nTotal tiles indexed   : {len(df_index)}")
print(f"Unique PointIDs       : {df_index['PointID'].nunique()}")
print(f"Quarters              : {sorted(df_index['Quarter'].unique())}")
print(f"Complete points (4/4) : "
      f"{(df_index.groupby('PointID').size() == 4).sum()}")
print(f"\nSample:")
print(df_index.head(8).to_string(index=False))

Scanning ZIP archives for tiles...

Total tiles indexed   : 4448
Unique PointIDs       : 1112
Quarters              : ['Q1', 'Q2', 'Q3', 'Q4']
Complete points (4/4) : 1112

Sample:
 PointID Quarter                                                                 zip_path                      tif_path
     181      Q1 /content/drive/MyDrive/Datamaraws_2025_Data/tiles_zipped/tiles_Aklan.zip 00181/point_00181_2025_Q1.tif
     181      Q2 /content/drive/MyDrive/Datamaraws_2025_Data/tiles_zipped/tiles_Aklan.zip 00181/point_00181_2025_Q2.tif
     181      Q3 /content/drive/MyDrive/Datamaraws_2025_Data/tiles_zipped/tiles_Aklan.zip 00181/point_00181_2025_Q3.tif
     181      Q4 /content/drive/MyDrive/Datamaraws_2025_Data/tiles_zipped/tiles_Aklan.zip 00181/point_00181_2025_Q4.tif
     182      Q1 /content/drive/MyDrive/Datamaraws_2025_Data/tiles_zipped/tiles_Aklan.zip 00182/point_00182_2025_Q1.tif
     182      Q2 /content/drive/MyDrive/Datamaraws_2025_Data/tiles_zipped/tiles_Aklan.zip 00182/poi

In [7]:
# ── IMAGE LOADER ───────────────────────────────────────
# CHANGED from phase2:
#   - phase2 parsed raw float32 bytes from TFRecord
#   - Here we read GeoTIFF directly from inside a ZIP using
#     rasterio's zip:// URI scheme (no extraction needed)
#   - Resize 1000×1000 → 224×224 (training tiles were already
#     224×224 after TFRecord conversion; these are 10km tiles)
#   - Same percentile normalization and VGG preprocessing
#     as training pipeline

def load_tile_from_zip(zip_path, tif_internal_path,
                       target_size=(224, 224)):
    """
    Read a GeoTIFF from inside a ZIP, normalize, resize
    to 224x224, and apply VGG16 preprocessing.
    Returns float32 array (224, 224, 3) or None on failure.
    """
    try:
        # rasterio can read directly from ZIP without extracting
        uri = f"zip://{zip_path}!/{tif_internal_path}"
        with rasterio.open(uri) as src:
            # Read first 3 bands (B4=Red, B3=Green, B2=Blue)
            # Matches training band order
            data = src.read([1, 2, 3]).astype(np.float32)

        # Transpose to HWC
        img = np.transpose(data, (1, 2, 0))

        # Replace nodata / NaN with 0
        img = np.nan_to_num(img, nan=0.0)

        # Percentile stretch normalization
        # Matches training TFRecord conversion exactly
        p2, p98 = np.percentile(img, (2, 98))
        if p98 <= p2:
            p98 = p2 + 1e-6
        img = np.clip((img - p2) / (p98 - p2), 0.0, 1.0)

        # Resize 1000×1000 → 224×224
        # CHANGED from phase2: training tiles were already 224×224
        # These 2025 tiles are 10km = 1000×1000px at 10m resolution
        img_tensor = tf.image.resize(
            img, target_size,
            method=tf.image.ResizeMethod.BILINEAR
        ).numpy()

        # VGG16 preprocessing
        # Same as training: scale to [0,255] then subtract ImageNet mean
        img_vgg = img_tensor * 255.0
        img_vgg = tf.keras.applications.vgg16\
                    .preprocess_input(img_vgg)

        return img_vgg.numpy() if hasattr(img_vgg, 'numpy') \
               else img_vgg

    except Exception as e:
        return None

# Quick sanity check on one tile
row0     = df_index.iloc[0]
test_img = load_tile_from_zip(row0['zip_path'],
                               row0['tif_path'])
if test_img is not None:
    print(f"✓ Tile loaded successfully")
    print(f"  Shape       : {test_img.shape}")
    print(f"  Pixel range : [{test_img.min():.2f}, "
          f"{test_img.max():.2f}]")
    print(f"  PointID     : {row0['PointID']}, "
          f"Quarter: {row0['Quarter']}")
else:
    print("✗ Tile load failed. Check ZIP path.")

✓ Tile loaded successfully
  Shape       : (224, 224, 3)
  Pixel range : [-123.68, 151.06]
  PointID     : 181, Quarter: Q1


In [8]:
# ── EXTRACTION LOOP ────────────────────────────────────
# CHANGED from phase2:
#   - phase2 iterated over a tf.data TFRecordDataset in batches
#   - Here we manually batch from df_index and read from ZIPs
#   - Key column is PointID (was ClusterID in phase2)
#   - After CNN extraction, apply PCA 4096→128
#   - Checkpointing saves to Drive every 50 batches

# Load checkpoint if resuming after disconnect
if os.path.exists(CHECKPOINT):
    df_done   = pd.read_csv(CHECKPOINT)
    done_keys = set(zip(
        df_done['PointID'].astype(int),
        df_done['Quarter'].astype(str)
    ))
    records   = df_done.to_dict('records')
    print(f"Resuming from checkpoint: {len(records)} done")
else:
    done_keys = set()
    records   = []
    print("Starting fresh extraction")

# Filter out already-done tiles
df_remaining = df_index[
    ~df_index.apply(
        lambda r: (int(r['PointID']), str(r['Quarter']))
                  in done_keys, axis=1
    )
].reset_index(drop=True)

print(f"Tiles remaining : {len(df_remaining)}")
print(f"Already done    : {len(done_keys)}")

start     = datetime.now()
batch_num = 0
n_failed  = 0

for batch_start in range(0, len(df_remaining), BATCH_SIZE):
    batch_end  = min(batch_start + BATCH_SIZE,
                     len(df_remaining))
    batch_rows = df_remaining.iloc[batch_start:batch_end]

    imgs      = []
    meta      = []

    for _, row in batch_rows.iterrows():
        img = load_tile_from_zip(
            row['zip_path'], row['tif_path'])
        if img is not None:
            imgs.append(img)
            meta.append((int(row['PointID']),
                         str(row['Quarter'])))
        else:
            n_failed += 1

    if not imgs:
        batch_num += 1
        continue

    imgs_arr = np.array(imgs, dtype=np.float32)

    # CNN feature extraction (4096-dim)
    # Same as phase2: feature_vector layer output
    feats_4096 = feature_extractor.predict(
        imgs_arr, verbose=0)

    # CHANGED: apply PCA 4096→128
    # phase2 had no PCA (it saved raw 4096-dim vectors)
    # Here we apply the saved PCA from final model training
    feats_pca = pca.transform(feats_4096)

    # Build records
    for j, (pid, quarter) in enumerate(meta):
        rec = {'PointID': pid, 'Quarter': quarter}
        rec.update({
            f'CNN_{k}': float(feats_pca[j][k])
            for k in range(N_COMPONENTS)
        })
        records.append(rec)
        done_keys.add((pid, quarter))

    batch_num += 1

    # Progress and checkpoint
    if batch_num % 10 == 0:
        elapsed = (datetime.now() - start).seconds
        print(f"  [{datetime.now().strftime('%H:%M:%S')}] "
              f"Batch {batch_num} | "
              f"Done: {len(records)} | "
              f"Failed: {n_failed} | "
              f"{elapsed//60}m {elapsed%60}s")

    if batch_num % 50 == 0:
        pd.DataFrame(records).to_csv(
            CHECKPOINT, index=False)
        print(f"  Checkpoint saved.")

print(f"\nExtraction complete.")
print(f"Records  : {len(records)}")
print(f"Failed   : {n_failed} tiles (load errors)")

Starting fresh extraction
Tiles remaining : 4448
Already done    : 0
  [20:27:32] Batch 10 | Done: 320 | Failed: 0 | 11m 12s
  [20:38:47] Batch 20 | Done: 640 | Failed: 0 | 22m 27s
  [20:50:29] Batch 30 | Done: 960 | Failed: 0 | 34m 9s
  [20:59:22] Batch 40 | Done: 1280 | Failed: 0 | 43m 2s
  [21:09:24] Batch 50 | Done: 1600 | Failed: 0 | 53m 4s
  Checkpoint saved.
  [21:18:07] Batch 60 | Done: 1920 | Failed: 0 | 61m 47s
  [21:26:45] Batch 70 | Done: 2240 | Failed: 0 | 70m 25s
  [21:36:00] Batch 80 | Done: 2560 | Failed: 0 | 79m 39s
  [21:46:01] Batch 90 | Done: 2880 | Failed: 0 | 89m 41s
  [21:54:24] Batch 100 | Done: 3200 | Failed: 0 | 98m 4s
  Checkpoint saved.
  [22:03:38] Batch 110 | Done: 3520 | Failed: 0 | 107m 18s
  [22:13:11] Batch 120 | Done: 3840 | Failed: 0 | 116m 51s
  [22:22:50] Batch 130 | Done: 4160 | Failed: 0 | 126m 29s

Extraction complete.
Records  : 4448
Failed   : 0 tiles (load errors)


In [9]:
# ── SAVE AND VERIFY ────────────────────────────────────
# CHANGED from phase2:
#   - Sorted by PointID + Quarter (was ClusterID + Quarter)
#   - Feature dims = N_COMPONENTS (128) not 4096
#   - Saved to Drive as 2025 output file
#   - Also copied to GCS for backup

df_final = pd.DataFrame(records).sort_values(
    ['PointID', 'Quarter']
).reset_index(drop=True)

# Save to Drive
df_final.to_csv(OUTPUT_DRIVE, index=False)
print(f"Saved to Drive: {OUTPUT_DRIVE}")

# Copy to GCS
import subprocess
subprocess.run([
    'gsutil', 'cp', OUTPUT_DRIVE,
    f'{GCS_OUTPUT}/dynamic_features_2025_prediction_points.csv'
])
print(f"Copied to GCS : {GCS_OUTPUT}")

# Verification
print(f"\n{'='*50}")
print("FEATURE EXTRACTION COMPLETE")
print(f"{'='*50}")
print(f"Total records    : {len(df_final)}")
print(f"Unique PointIDs  : {df_final['PointID'].nunique()}")
print(f"Quarters         : "
      f"{sorted(df_final['Quarter'].unique())}")
print(f"Feature dims     : "
      f"{df_final.shape[1] - 2} (PCA-reduced, should be {N_COMPONENTS})")
print(f"Load failures    : {n_failed} tiles skipped")

# Check completeness (all 4 quarters per point)
counts = df_final.groupby('PointID').size()
complete = (counts == 4).sum()
incomplete = (counts < 4).sum()
print(f"\nComplete points (4/4 quarters) : {complete}")
print(f"Incomplete points (<4 quarters): {incomplete}")
if incomplete > 0:
    print("Incomplete PointIDs:")
    print(counts[counts < 4].to_string())

# Spot check feature values
print(f"\nFeature value sample (first 5 dims):")
sample_cols = ['PointID', 'Quarter',
               'CNN_0', 'CNN_1', 'CNN_2', 'CNN_3', 'CNN_4']
print(df_final[sample_cols].head(4).to_string(index=False))

print(f"\nNext: run feature merge notebook to assemble")
print(f"X_dynamic (N, 4, {N_COMPONENTS}), X_static (N, 20)")
print(f"then run LSTM inference.")

Saved to Drive: /content/drive/MyDrive/Datamaraws_2025_Data/dynamic_features_2025_prediction_points.csv
Copied to GCS : gs://tala-sentinel2-data/features/output_2025

FEATURE EXTRACTION COMPLETE
Total records    : 4448
Unique PointIDs  : 1112
Quarters         : ['Q1', 'Q2', 'Q3', 'Q4']
Feature dims     : 256 (PCA-reduced, should be 256)
Load failures    : 0 tiles skipped

Complete points (4/4 quarters) : 1112
Incomplete points (<4 quarters): 0

Feature value sample (first 5 dims):
 PointID Quarter      CNN_0     CNN_1     CNN_2    CNN_3     CNN_4
       1      Q1  -9.565256 12.873568 -4.560822 7.119608 -0.102740
       1      Q2  -5.095402 19.444862 -5.929778 9.206253 -1.020484
       1      Q3 -16.927841  4.182848 -6.925879 6.073661  0.174612
       1      Q4 -21.098097 -2.664571 -6.048522 3.595191  0.733179

Next: run feature merge notebook to assemble
X_dynamic (N, 4, 256), X_static (N, 20)
then run LSTM inference.
